# Foundation 02 — Threat Modeling Agentic Systems

Convert a versioned agent architecture into reviewed threat records whose important paths are bound to deterministic controls, executable tests, bounded telemetry, accountable owners, and explicit residual-risk decisions. Model output remains an untrusted discovery aid, never review authority.

![Agentic threat modeling from architecture to assurance](architecture.svg)

The workflow follows OWASP's four threat-modeling questions. Suggestions help discovery; authenticated review and deterministic evidence decide what enters the model and when architecture drift requires refresh.

## 1. Load the course lab

The notebook imports the reusable course module rather than copying its security logic.

In [ ]:
import runpy
from datetime import datetime, timedelta, timezone
from dataclasses import replace
import sys
sys.path.insert(0, '.')
ns = runpy.run_path('lab.py')
ThreatProposal, ReviewerContext = ns['ThreatProposal'], ns['ReviewerContext']
admit_proposal, build_complete_model = ns['admit_proposal'], ns['build_complete_model']
review_threat_model, unsafe_row_count_baseline = ns['review_threat_model'], ns['unsafe_row_count_baseline']
model = build_complete_model()

## 2. Establish the safe baseline

Observe the trusted inputs and the decision evidence before injecting failures.

In [ ]:
review = review_threat_model(model)
assert review.accepted and not review.blockers
assert review.required_flow_coverage == review.high_risk_control_coverage == 1
assert review.high_risk_test_coverage == review.high_risk_telemetry_coverage == 1
{'architecture_version': model.architecture.version, 'records': review.records, 'required_flows': len(model.architecture.required_flow_ids), 'accepted': review.accepted}

## 3. Inject an attack

Change one security-relevant boundary and keep the rest of the fixture stable.

In [ ]:
first = model.records[0]
unsafe = replace(model, records=(replace(first, control_ids=('C404',), reviewer_id='model:planner'), *model.records[1:]))
baseline_accepts = unsafe_row_count_baseline(unsafe)
strict_review = review_threat_model(unsafe)
assert baseline_accepts and not strict_review.accepted
{'row_count_baseline': baseline_accepts, 'assurance_gate': strict_review.accepted, 'blockers': strict_review.blockers}

## 4. Attempt a bypass

The assertions below make the security property executable and regression-testable.

In [ ]:
proposal = ThreatProposal(**{field: getattr(first, field) for field in ThreatProposal.__dataclass_fields__})
model_reviewer = ReviewerContext('model:planner', frozenset({'security-reviewer'}), authenticated=False)
human_reviewer = ReviewerContext('reviewer:8', frozenset({'security-reviewer'}))
rejected = admit_proposal(proposal, model_reviewer, residual_risk='model says low')
admitted = admit_proposal(proposal, human_reviewer, residual_risk='accepted by owner; monitor the decision path')
assert rejected.reason == 'reviewer-unauthenticated'
assert admitted.accepted and admitted.record.reviewer_id == 'reviewer:8'
(rejected, admitted)

## 5. Evaluate observable outcomes

Use explicit denominators or counts. Private model reasoning is neither required nor recorded.

In [ ]:
report, cases = ns['evaluate_models']()
assert (report.cases, report.valid_cases, report.negative_cases) == (7, 1, 6)
assert report.unexpected_acceptance_rate == 0 and report.valid_model_acceptance_rate == 1
{'populations': {'all': report.cases, 'valid': report.valid_cases, 'negative': report.negative_cases}, 'rates': {'unexpected_acceptance': report.unexpected_acceptance_rate, 'valid_acceptance': report.valid_model_acceptance_rate, 'flow_coverage': report.required_flow_coverage, 'high_risk_control_coverage': report.high_risk_control_coverage, 'high_risk_test_coverage': report.high_risk_test_coverage, 'high_risk_telemetry_coverage': report.high_risk_telemetry_coverage}, 'case_outcomes': {case.name: case.result.accepted for case in cases}}

## 6. Exercise a second failure mode

In [ ]:
stale = review_threat_model(replace(model, architecture_version='northwind-agent-1'))
missing_flow = review_threat_model(replace(model, records=model.records[:-1]))
assert 'stale-architecture-version' in stale.blockers
assert 'unmodeled-flow:F7-memory' in missing_flow.blockers
(stale.blockers, missing_flow.blockers)

## 7. Executable ANY/ALL attack tree

Attack-tree leaves are preconditions, not threat-record counts. Minimal cut sets expose the smallest combinations that satisfy each path so controls can break at least one required leaf.

In [ ]:
tree = ns['build_exfiltration_tree']()
cuts = ns['minimal_cut_sets'](tree)
assert len(cuts) == 3
assert all(ns['attack_succeeds'](tree, cut) for cut in cuts)
sorted(tuple(sorted(cut)) for cut in cuts)

## 8. Residual risk cannot improve by assertion

The 1–5 values are ordinal prioritization aids, not calibrated probabilities. A worse residual score and a missing verification fixture both block review.

In [ ]:
magic = review_threat_model(replace(model, records=(replace(first, residual_likelihood=5, residual_impact=5), *model.records[1:])))
untested = review_threat_model(replace(model, records=(replace(first, test_ids=()), *model.records[1:])))
assert 'residual-exceeds-inherent:T1' in magic.blockers
assert 'test-integrity:T1' in untested.blockers
(magic.blockers, untested.blockers)

## 9. OWASP pytm 1.4: a real code-first DFD

The adapter builds the same nine elements and seven flows with real pytm Actor, Agent, LLM, Server, Datastore, Boundary, and Dataflow primitives. The SDK structures discovery; application review still decides acceptance.

In [ ]:
pytm = runpy.run_path('pytm_adapter.py')
summary = pytm['demo']()
assert summary.element_count == 9 and summary.flow_count == 7
assert {'Actor','Agent','LLM','Server','Datastore'} <= set(summary.element_types)
{'elements': summary.element_count, 'flows': summary.flow_count, 'types': summary.element_types}

## 10. Production replacement

Production replacement: inventory every deployed service, model, identity, queue, store, tool, MCP server, credential, region, data flow, authority flow, and external effect; version diagrams and policy with deployed artifacts; authenticate reviewers; use durable workflow and risk registers; map verified controls to CI/CD and runtime evidence; protect telemetry and sensitive model data; assign owners and expiry; trigger review on architecture, tool, identity, model, memory, oversight, consequence, or recovery changes; and reconcile threat-model evidence with incident and vulnerability management. The local dataclasses, ordinal scores, static references, and pytm DFD demonstrate traceability mechanics only.

## 11. Exercises

1. Add a model-provider flow and distinguish provider authentication from end-user authorization.
2. Extend the attack tree with a supply-chain path and identify its minimal cut set.
3. Add a privacy threat using LINDDUN and explain where STRIDE is insufficient.
4. Add an availability threat and a tested degradation control.
5. Change a flow or boundary version and prove review blocks stale evidence.
6. Compare the same model in Threat Dragon, Threagile, Microsoft TMT, or pytm without outsourcing risk acceptance to the tool.

## Checkpoint

Explain which trusted component enforces the invariant, what evidence proves the decision, and what residual risk remains.